# Iterative Title Generation and Optimization

This notebook uses the trained numeric inference models and qualitative LLM analysis to iteratively improve video titles for specific channels.

**Export Path:** `/content/drive/MyDrive/numeric_inference_outputs/title_optimization_results.json`  
**Type:** JSON (List of objects containing original metadata, best title, and full iteration history)

In [3]:
# Setup and Dependencies
# We only upgrade google-generativeai to avoid NumPy 2.0 conflicts in Colab
!pip install -q google-generativeai==0.8.3
!pip install -q sentence-transformers

import os
import json
import time
import numpy as np
import pandas as pd
import hashlib
from google.colab import drive, userdata
import google.generativeai as genai
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer

drive.mount('/content/drive')

GEMINI_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)
MODEL_NAME = 'gemini-3.1-flash-lite'
EMBEDDING_MODEL_NAME = 'all-MiniLM-L6-v2'

BASE_PATH = '/content/drive/MyDrive/numeric_inference_outputs/'
EVAL_DATA_PATH = os.path.join(BASE_PATH, 'top_significant_channels_eval.json')
LLM_RESULTS_PATH = os.path.join(BASE_PATH, 'llm_analysis_results.json')
EMBEDDING_CACHE_PATH = os.path.join(BASE_PATH, 'video_title_embeddings_latest.json')
TRAIN_DATA_PATH = os.path.join(BASE_PATH, 'train_structured_latest.json')
GENERATION_CACHE_PATH = os.path.join(BASE_PATH, 'title_generation_cache.json')
FINAL_RESULTS_PATH = os.path.join(BASE_PATH, 'title_optimization_results.json')

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
# Data Loading
with open(EVAL_DATA_PATH, 'r') as f:
    eval_dataset = json.load(f)

with open(LLM_RESULTS_PATH, 'r') as f:
    llm_analysis = json.load(f)

with open(EMBEDDING_CACHE_PATH, 'r') as f:
    embedding_cache = json.load(f)

with open(TRAIN_DATA_PATH, 'r') as f:
    train_data = json.load(f)

def load_gen_cache():
    if os.path.exists(GENERATION_CACHE_PATH):
        with open(GENERATION_CACHE_PATH, 'r') as f:
            return json.load(f)
    return {}

def save_gen_cache(cache):
    with open(GENERATION_CACHE_PATH, 'w') as f:
        json.dump(cache, f, indent=4)

generation_cache = load_gen_cache()
print("Data and generation cache loaded successfully.")

Data and generation cache loaded successfully.


In [5]:
# Reconstruct global PCA from training data
all_train_embeddings = []
for channel in train_data:
    for video in channel['videos']:
        all_train_embeddings.append(embedding_cache[video['title']])

X_train = np.array(all_train_embeddings)
pca = PCA(n_components=15, random_state=42)
pca.fit(X_train)

print(f"Reconstructed PCA with {pca.n_components_} components.")

Reconstructed PCA with 15 components.


In [6]:
# Utility functions for prediction
def predict_views(titles, channel_id, channel_models_params):
    """
    Predict log-views for a list of titles.
    """
    # 1. Embed titles
    embs = embedding_model.encode(titles)

    # 2. Project to PCA
    reduced = pca.transform(embs)

    # 3. Predict using parameters
    # Data schema from inference notebook:
    # intercept: float(model.params[0])
    # coefficients: model.params[1:].tolist()
    params = channel_models_params[channel_id]
    intercept = params['intercept']
    coeffs = np.array(params['coefficients'])

    predictions = (reduced @ coeffs) + intercept

    return predictions.tolist()

In [7]:
# Configuration for target channels
TARGET_CHANNELS_REQUESTED = ["20VC with Harry Stebbings", "A16z"]
target_channel_data = []
channel_models_params = {}

# Case-insensitive matching
target_channels_lower = [c.lower() for c in TARGET_CHANNELS_REQUESTED]

for channel in eval_dataset:
    if channel['channel_name'].lower() in target_channels_lower:
        target_channel_data.append(channel)
        channel_models_params[channel['channel_id']] = {
            'coefficients': channel['model']['coefficients'],
            'intercept': channel['model']['intercept']
        }

print(f"Targeted {len(target_channel_data)} channels.")

Targeted 2 channels.


In [8]:
def get_gemini_response(prompt):
    prompt_hash = hashlib.md5(prompt.encode()).hexdigest()
    if prompt_hash in generation_cache:
        return generation_cache[prompt_hash]

    model = genai.GenerativeModel(MODEL_NAME)
    for _ in range(3):
        try:
            response = model.generate_content(prompt)
            text = response.text
            generation_cache[prompt_hash] = text
            save_gen_cache(generation_cache)
            time.sleep(5)
            return text
        except Exception as e:
            print(f"Error: {e}. Retrying...")
            time.sleep(5)
    return ""

def parse_titles(text):
    """Extract 10 titles from LLM response."""
    lines = text.strip().split('\n')
    titles = []
    for line in lines:
        line = line.strip()
        if not line: continue
        # Strip leading numbers or bullets
        if line[0].isdigit() or line[0] in ['-', '*']:
            parts = line.split('. ', 1) if '.' in line else line.split(' ', 1)
            if len(parts) > 1:
                titles.append(parts[1].strip().strip('"'))
            else:
                titles.append(line.strip('-* ').strip('"'))
        else:
            titles.append(line.strip('"'))
        if len(titles) == 10: break
    return titles[:10]

In [9]:
# Iterative Optimization Loop
results = []

for channel in target_channel_data:
    cid = channel['channel_id']
    cname = channel['channel_name']

    global_desc = llm_analysis['global_performance_descriptions'].get(cid, "")
    dim_analysis = llm_analysis['channel_significant_dimension_analysis'].get(cid, "")

    # Select 10 test videos
    test_vids = channel['test_videos'][:10]

    for vid in test_vids:
        base_title = vid['title']
        base_pred = float(np.log1p(vid['actual_views']))

        print(f"\nOptimizing title for {cname}: '{base_title}'")

        history = [] # To store list of {iteration, titles: [{text, score}]}

        for i in range(5):
            print(f"  Iteration {i+1}...")

            # Build Prompt
            best_so_far = max([base_pred] + [max([t['score'] for t in h['titles']]) for h in history]) if history else base_pred
            prompt = f"""You are an expert YouTube title strategist for the channel '{cname}'.
Channel success drivers:
{global_desc}

Semantic performance analysis:
{dim_analysis}

Original Title: {base_title}
Current best predicted performance (log-views): {best_so_far:.4f}
"""
            if history:
                # Include top 3 and bottom 3 from previous iterations for feedback
                all_prev = []
                for h in history:
                    for t_obj in h['titles']:
                        all_prev.append(t_obj)
                all_prev.sort(key=lambda x: x['score'], reverse=True)

                prompt += "\nPrevious suggestions feedback:\n"
                prompt += "Top performing suggestions:\n"
                for t_obj in all_prev[:3]:
                    prompt += f"- {t_obj['text']} (Score: {t_obj['score']:.4f})\n"

                prompt += "Lower performing suggestions (Avoid these patterns):\n"
                for t_obj in all_prev[-3:]:
                    prompt += f"- {t_obj['text']} (Score: {t_obj['score']:.4f})\n"

            prompt += "\nTask: Generate 10 new, improved titles for this video that will maximize views based on the channel's success drivers and semantic analysis. Return ONLY the 10 titles, one per line."

            llm_text = get_gemini_response(prompt)
            new_titles = parse_titles(llm_text)

            if not new_titles:
                 print("    Error: No titles generated.")
                 continue

            preds = predict_views(new_titles, cid, channel_models_params)

            iter_titles = []
            for t, p in zip(new_titles, preds):
                iter_titles.append({'text': t, 'score': p})

            history.append({
                'iteration': i + 1,
                'titles': iter_titles
            })

            # Identify best this round
            best_idx = np.argmax(preds)
            diff = preds[best_idx] - base_pred
            sign = "+" if diff >= 0 else ""
            print(f"    Best this round: '{new_titles[best_idx]}' ({sign}{diff:.4f} log-views)")

        # Collect final results for this video
        all_optimized = []
        for h in history:
            all_optimized.extend(h['titles'])

        all_optimized.sort(key=lambda x: x['score'], reverse=True)
        best_title_obj = all_optimized[0]

        results.append({
            'channel': cname,
            'video_id': vid['video_id'],
            'original_title': base_title,
            'original_score': base_pred,
            'best_optimized_title': best_title_obj['text'],
            'best_optimized_score': best_title_obj['score'],
            'improvement': best_title_obj['score'] - base_pred,
            'improvement_pct': (np.expm1(best_title_obj['score']) - np.expm1(base_pred)) / np.expm1(base_pred) * 100 if np.expm1(base_pred) > 0 else 0,
            'history': history
        })

print("\nOptimization complete.")


Optimizing title for a16z: 'Udio: From Text to Tune'
  Iteration 1...
    Best this round: 'The Future of Sound: Inside the AI Startup Disrupting the $25B Music Industry' (+1.7610 log-views)
  Iteration 2...
    Best this round: 'Why Music is the Next Frontier in the Billion-Dollar AI Arms Race' (+1.6641 log-views)
  Iteration 3...
    Best this round: 'Inside the AI Startup Changing How We Build Billion-Dollar Media Hits' (+2.3825 log-views)
  Iteration 4...
    Best this round: 'Udio CEO: Why AI Will Make Every Creator a One-Person Media Empire' (+2.3492 log-views)
  Iteration 5...
    Best this round: 'Udio CEO: How AI is Orchestrating the Next $100B Media Empire' (+2.4300 log-views)

Optimizing title for a16z: 'Marc Andreessen & Amjad Masad on “Good Enough” AI, AGI, and the End of Coding'
  Iteration 1...
    Best this round: 'The End of Software Development: Marc Andreessen on the AI Takeover' (-0.5609 log-views)
  Iteration 2...
    Best this round: 'Beyond Code: Marc Andreessen

In [10]:
# Export final results
with open(FINAL_RESULTS_PATH, 'w') as f:
    json.dump(results, f, indent=4)
print(f"Final results exported to {FINAL_RESULTS_PATH}")

Final results exported to /content/drive/MyDrive/numeric_inference_outputs/title_optimization_results.json


## Final Report

In [11]:
df_results = pd.DataFrame(results)

# Use the matched channel names for reporting
target_channel_names = [c['channel_name'] for c in target_channel_data]
for channel in target_channel_names:
    print(f"\n=== {channel} Summary ===")
    chan_df = df_results[df_results['channel'] == channel]
    for _, row in chan_df.iterrows():
        print(f"Original: {row['original_title']}")
        print(f"Best:     {row['best_optimized_title']}")
        print(f"Expected Improvement: {row['improvement']:.4f} log-views ({row['improvement_pct']:.2f}%)")
        print("-" * 20)

print(f"\nAverage Improvement across all videos: {df_results['improvement'].mean():.4f} log-views")


=== a16z Summary ===
Original: Udio: From Text to Tune
Best:     Udio CEO: How AI is Orchestrating the Next $100B Media Empire
Expected Improvement: 2.4300 log-views (1036.28%)
--------------------
Original: Marc Andreessen & Amjad Masad on “Good Enough” AI, AGI, and the End of Coding
Best:     Marc Andreessen: Why "Good Enough" AI Is a Threat to Silicon Valley
Expected Improvement: -0.4692 log-views (-37.45%)
--------------------
Original: The 2045 Superintelligence Timeline: Epoch AI’s Data-Driven Forecast
Best:     The Singularity Race: Why America Must Own the AI Infrastructure or Perish
Expected Improvement: 1.3523 log-views (286.66%)
--------------------
Original: AI & The Future of Modern Warfare
Best:     From Silicon Valley to the Frontline: The AI Race That Will Define the Century
Expected Improvement: 1.1554 log-views (217.56%)
--------------------
Original: Mark Cuban on the NBA, Cost Plus Drugs, and How to Fix Politics
Best:     Mark Cuban on How AI and Big Pharma Are Col